In [1]:
from omegaconf import OmegaConf
from jupyterscad import view

from solid2 import square, cube, cylinder

from itertools import accumulate

from configuration import schema, ConfigSchema
from connectors import generate_male_connector, generate_female_connector
from common import generate_keys_row

In [2]:
yaml_config = OmegaConf.load("default.conf.yaml")
conf = OmegaConf.merge(schema, yaml_config)
conf

{'dist_u': 19.05, 'white_key_dims': {'length': 1.75, 'width': 1.25}, 'black_key_dims': {'length': 1.75, 'width': 1.0}, 'rows_height_diff_mm': 12.0, 'mount_plate_width': 1.25, 'mount_u': 14.0, 'keycap_u': 18.0, 'keycap_height_mm': 1.5, 'keycap_rounding_corner_mm': 2.0, 'output_dir': PosixPath('build'), 'white_black_keys_offset_mm': 4.0, 'base_height_mm': 8.0, 'connector_dims': {'base_diff_mm': 3.0, 'margin_mm': 0.06}, 'stand_r_mm': 5.0, 'stand_density': 2}

In [ ]:
def generate_octave(white_keys: int, conf: ConfigSchema):
    assert white_keys <= 7
    white_keys_to_black_num_mapping = {
        1: 0,
        2: 1,
        3: 2,
        4: 2,
        5: 3,
        6: 4,
        7: 5,
    }

    wk_total_width = conf.white_key_dims.width * conf.dist_u
    octave_width = wk_total_width * white_keys
    white_plate_len = conf.white_key_dims.length * conf.dist_u
    w_distances = [(wk_total_width - conf.mount_u) / 2] + [wk_total_width] * 7
    white_mount_plate = (
        # upper wall, with mx mounting holes
        generate_keys_row(
            octave_width,
            white_plate_len,
            w_distances[:white_keys],
            (white_plate_len - conf.mount_u) / 2,
            conf,
        )
        # front wall of the keyboard
        + cube([octave_width, conf.mount_plate_width, conf.base_height_mm]).down(
            conf.base_height_mm
        )
        # connectors
        + generate_female_connector(w_distances[0], white_plate_len, conf)
        + generate_male_connector(w_distances[0], white_plate_len, conf).translateX(
            octave_width
        )
    )

    for i in range(1, white_keys, conf.stand_density):
        white_mount_plate += cylinder(
            h=conf.base_height_mm, r=conf.stand_r_mm
        ).translate(
            [
                i * wk_total_width,
                white_plate_len - conf.stand_r_mm,
                -conf.base_height_mm,
            ]
        )

    bw_diff = conf.white_black_keys_offset_mm + conf.mount_plate_width
    b_distances = [
        wk_total_width - conf.mount_u / 2,
        wk_total_width,
        wk_total_width * 2,
        wk_total_width,
        wk_total_width,
    ]
    black_mount_plate = (
        generate_keys_row(
            octave_width,
            conf.dist_u,
            b_distances[: white_keys_to_black_num_mapping[white_keys]],
            (conf.dist_u - conf.mount_u) / 2,
            conf,
        )
        + cube([octave_width, conf.mount_plate_width, bw_diff]).down(bw_diff)
        + cube([octave_width, conf.mount_plate_width, bw_diff + conf.base_height_mm])
        .down(bw_diff + conf.base_height_mm)
        .translateY(conf.dist_u - conf.mount_plate_width)
        + generate_female_connector(b_distances[0], conf.dist_u, conf)
        + generate_male_connector(b_distances[0], conf.dist_u, conf).translateX(
            octave_width
        )
    )

    return black_mount_plate + white_mount_plate.translate(
        [
            0,
            -white_plate_len,
            -conf.white_black_keys_offset_mm - conf.mount_plate_width,
        ]
    )


view(generate_octave(7, conf))

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.7, position=(3.0, 5.0,…

In [ ]:
# TODO: each octave is indivisible part
#       but it is possible to generate half of the octave
#       currently I have 35 switches, so the max I can get is 2.5 octaves.
#       30~ keys + 5 mods (octave up, octave down, maybe some play, record, etc)

